In [ ]:
import numpy as np
import tensorflow as tf


path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt', 
    'https://googleapis.com'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f"Total Length of Text : {len(text)} characters")
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
print(f"Vocabulary Size : {vocab_size} Unique Characters")

char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)
text_as_int = np.array([char2idx[c] for c in text])

seq_length = 100 
examples_per_epoch = len(text) // (seq_length + 1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

embedding_dim = 256
rnn_units = 1024

class TextGeneratorModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(
            rnn_units,
            return_sequences=True,
            return_state=True
        )
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = inputs
        x = self.embedding(x, training=training)
        
        if states is None:
            states = self.lstm.get_initial_state(x)
            
        x, memory_state, carry_state = self.lstm(x, initial_state=states, training=training)
        x = self.dense(x, training=training)

        if return_state:
            return x, [memory_state, carry_state]
        return x

model = TextGeneratorModel(vocab_size=vocab_size, embedding_dim=embedding_dim, rnn_units=rnn_units)

model.compile(
    optimizer='adam', 
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

EPOCHS = 3  
model.fit(dataset, epochs=EPOCHS)

class OneStep(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        skip_ids = [self.ids_from_chars.get('[UNK]', -1)]
        sparse_mask = tf.sparse.SparseTensor(
            values=[-float('inf')] * len(skip_ids),
            indices=np.array([[i, 0] for i in skip_ids]),
            dense_shape=[vocab_size, 1]
        )
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)

    @tf.function
    def generate_one_step(self, inputs, states=None):
        input_ids = tf.convert_to_tensor([[self.ids_from_chars[c] for c in inputs[0]]])
        
        logits, states = self.model(inputs=input_ids, states=states, return_state=True)
        logits = logits[:, -1, :]
        logits = logits / self.temperature
        logits = logits + self.prediction_mask

        predicted_ids = tf.random.categorical(logits, num_samples=1)
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        predicted_char = self.chars_from_ids[predicted_ids[0].numpy()]
        return predicted_char, states

one_step_model = OneStep(model, idx2char, char2idx, temperature=0.7)

while True:
    start_string = input("\n-- Enter a Seed String (or type 'exit' to stop): ").strip()
    if start_string.lower() == 'exit' or not start_string:
        print("Exiting Generative Sequence Loop ...")
        break
        
    num_chars_to_generate = 250
    states = None
    next_char = start_string
    result = [next_char]

    for _ in range(num_chars_to_generate):
        next_char, states = one_step_model.generate_one_step([next_char], states=states)
        result.append(next_char)

    print("".join(result))